In [ ]:
import os
import glob
import h5py
import torch
import librosa
import numpy as np
import textgrid
import torch.nn.functional as F
from transformers import Wav2Vec2FeatureExtractor, HubertModel

# ====== Configuration ======
AUDIO_DIR = "../data/raw_data/xie21"

# Define configurations for both base and fine-tuned models
MODEL_CONFIGS = [
    {
        "model_id": "facebook/hubert-large-ll60k",
        "feat_output": "xie21_features.h5",
        "tsne_output": "xie21_tsne_3d.h5"
    },
    {
        "model_id": "facebook/hubert-large-ls960-ft",
        "feat_output": "xie21_features_ft.h5",
        "tsne_output": "xie21_tsne_3d_ft.h5"
    }
]

set1_list = [0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
set2_list = [17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]
combined_set = set1_list + set2_list

layers_spec = {
    "cnn": [2, 3, 4, 5, 6],
    "tr":  [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24], 
}
# ========================

def _register_cnn_hooks(hubert_model):
    store = {}
    handles = []
    conv_layers = hubert_model.feature_extractor.conv_layers
    def make_hook(i):
        def hook(module, inputs, output):
            store[i] = output.detach()
        return hook
    for i, layer in enumerate(conv_layers):
        handles.append(layer.register_forward_hook(make_hook(i)))
    return handles, store

def _cleanup_hooks(handles):
    for h in handles:
        try:
            h.remove()
        except:
            pass

def _extract_selected_layers(forward_module, input_values, layers_spec):
    cnn_handles, cnn_store = _register_cnn_hooks(forward_module)
    with torch.no_grad():
        out = forward_module(input_values, output_hidden_states=True, return_dict=True)
        tr_hidden = out.hidden_states
    _cleanup_hooks(cnn_handles)

    results = {}
    
    # Identify target temporal dimension: Sequence length of Transformer hidden layers
    # out.hidden_states[0] has shape (B, T_target, D)
    T_target = tr_hidden[0].size(1) 
    
    if "cnn" in layers_spec:
        for i in layers_spec["cnn"]:
            if i in cnn_store:
                feat = cnn_store[i]  # Original shape: (B, C, T_current)
                
                # Core Alignment Logic: Utilize 1D Adaptive Average Pooling to downsample T_current to T_target.
                # This achieves precise temporal alignment across all CNN layers with minimal information loss.
                if feat.size(2) != T_target:
                    feat = F.adaptive_avg_pool1d(feat, T_target)
                
                feat = feat.squeeze(0).permute(1, 0).contiguous() # Convert to (T_target, C)
                results[f"cnn_{i}"] = feat.cpu().numpy()
                
    if "tr" in layers_spec:
        for idx in layers_spec["tr"]:
            feat = tr_hidden[idx].squeeze(0) # (T_target, D)
            results[f"tr_{idx}"] = feat.cpu().numpy()
            
    return results

# ====== Execution Logic ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

audio_paths = glob.glob(os.path.join(AUDIO_DIR, "*.wav"))
if not audio_paths:
    print("Error: Audio files not found. Please verify the AUDIO_DIR path.")
else:
    for config in MODEL_CONFIGS:
        model_id = config["model_id"]
        output_h5 = config["feat_output"]
        
        print(f"\n--- Initializing Model: {model_id} ---")
        processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
        model = HubertModel.from_pretrained(model_id).to(device)
        model.eval()

        with h5py.File(output_h5, "w") as h5f:
            for each_path in audio_paths:
                speaker_id = os.path.basename(each_path).replace(".wav", "")
                
                if speaker_id not in h5f:
                    speaker_group = h5f.create_group(speaker_id)
                else:
                    speaker_group = h5f[speaker_id]
                
                audio, sr = librosa.load(each_path, sr=None, mono=True)
                wave_res = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                tg = textgrid.TextGrid.fromFile(each_path[:-3] + ".TextGrid")
                tg_sentence = tg[0]
                
                for idx, itv in enumerate(tg_sentence):
                    if itv.mark != "":
                        tg_sentence[idx - 1].maxTime = tg_sentence[idx].minTime
                tg_sentence = [itv for itv in tg_sentence if itv.mark != ""]
                
                for s_idx in combined_set:
                    each_sentence = tg_sentence[s_idx]
                    start = int(each_sentence.minTime * 16000)
                    end   = int(each_sentence.maxTime * 16000)
                    
                    inputs = processor(wave_res[start:end], sampling_rate=16000, return_tensors="pt", padding=False)
                    input_values = inputs.input_values.to(device)
                    
                    layer_feats = _extract_selected_layers(model, input_values, layers_spec)
                    
                    sentence_key = f"sentence_{s_idx:02d}"
                    if sentence_key in speaker_group:
                        del speaker_group[sentence_key]
                    sentence_group = speaker_group.create_group(sentence_key)
                    
                    for layer_key, feat_np in layer_feats.items():
                        sentence_group.create_dataset(layer_key, data=feat_np, compression="gzip")
                        
                print(f"Successfully processed speaker: {speaker_id}")
                
        print(f"Feature extraction completed for model {model_id}!")


In [ ]:
import h5py
import numpy as np
from joblib import Parallel, delayed
from sklearn.manifold import TSNE
import multiprocessing
import time

TSNE_DIM = 3

# Reuse MODEL_CONFIGS from previous cell
# It is assumed that MODEL_CONFIGS is defined in the workspace

def get_all_layers(h5_path):
    layers = set()
    with h5py.File(h5_path, "r") as h5f:
        first_speaker = list(h5f.keys())[0]
        first_sentence = list(h5f[first_speaker].keys())[0]
        for layer_key in h5f[first_speaker][first_sentence].keys():
            layers.add(layer_key)
    return list(layers)

def process_single_layer(layer_name, input_h5):
    """Extract data for a single layer and perform t-SNE reduction (joblib worker function)"""
    all_frames = []
    metadata = [] 
    
    with h5py.File(input_h5, "r") as h5f:
        for speaker_id in h5f.keys():
            for sentence_id in h5f[speaker_id].keys():
                if layer_name in h5f[speaker_id][sentence_id]:
                    feat_matrix = h5f[speaker_id][sentence_id][layer_name][:]
                    T = feat_matrix.shape[0]
                    all_frames.append(feat_matrix)
                    metadata.append({"speaker": speaker_id, "sentence": sentence_id, "length": T})
                    
    if not all_frames:
        return layer_name, None, metadata

    flatten = np.vstack(all_frames)
    print(f"[{layer_name}] Initiating t-SNE computation... Total frames: {flatten.shape[0]}, Feature dimension: {flatten.shape[1]}")
    start_time = time.time()
    
    tsne = TSNE(n_components=TSNE_DIM, random_state=42, n_jobs=1)
    reduced_features = tsne.fit_transform(flatten)
    
    cost_time = time.time() - start_time
    print(f"[{layer_name}] t-SNE computation finalized. Elapsed time: {cost_time:.2f} seconds")
    
    # Return dimension-reduced results to the main process
    return layer_name, reduced_features, metadata

# ========= Initialize Parallel t-SNE Computation =========
max_workers = min(multiprocessing.cpu_count(), 24)
print(f"Initializing parallel processing framework via joblib. Allocated CPU cores: {max_workers}")

for config in MODEL_CONFIGS:
    input_h5 = config["feat_output"]
    output_h5 = config["tsne_output"]
    
    print(f"\n--- Initiating Dimensionality Reduction for Configuration: {config['model_id']} ---")
    layers = get_all_layers(input_h5)
    print(f"Target layers for processing: {layers}")

    # Core execution: utilizing robust joblib concurrency
    # n_jobs specifies the degree of parallelism
    results = Parallel(n_jobs=max_workers)(
        delayed(process_single_layer)(layer, input_h5) for layer in layers
    )

    # Aggregate and write results to the output HDF5 structure
    print(f"\nAggregating results and writing to {output_h5}...")
    with h5py.File(output_h5, "w") as out_h5:
        for layer_name, reduced_features, metadata in results:
            if reduced_features is None: 
                continue
            
            layer_group = out_h5.create_group(layer_name)
            current_idx = 0
            for meta in metadata:
                spk, sen, length = meta["speaker"], meta["sentence"], meta["length"]
                sentence_3d = reduced_features[current_idx : current_idx + length]
                current_idx += length
                
                if spk not in layer_group:
                    layer_group.create_group(spk)
                layer_group[spk].create_dataset(sen, data=sentence_3d)

    print(f"Dimensionality reduction completed successfully. Results stored in {output_h5}")

print("\nPipeline execution fully completed.")
